In [0]:
# ================================================
# CELL 1 - CONFIGURATION
# ================================================
CATALOG = "crime_data"
OUTPUT_PATH = f"/Volumes/{CATALOG}/gold/outputs/final_reporting_dataset"

print(f"✅ Catalog: {CATALOG}")
print(f"✅ Output path: {OUTPUT_PATH}")

In [0]:
# ================================================
# CELL 2 - READ FROM GOLD TABLE
# ================================================
import pandas as pd
from pyspark.sql.functions import col, sum as spark_sum

df_final = spark.table("crime_data.gold.gold_final_dataset")

print(f"✅ Rows loaded: {df_final.count():,}")
print(f"✅ Columns: {df_final.columns}")

In [0]:
# ================================================
# CELL 3 - FINAL VALIDATION
# ================================================

print("=== NULL CHECK ON KEY COLUMNS ===")
key_cols = [
    "lsoa_code", "lsoa_name", "force_name",
    "year", "month_num"
]
for field in key_cols:
    null_count = df_final.filter(col(field).isNull()).count()
    status = "✅" if null_count == 0 else "⚠️"
    print(f"{status} {field}: {null_count:,} nulls")

print("\n=== YEAR RANGE CHECK ===")
df_final.groupBy("year") \
    .count() \
    .orderBy("year") \
    .show()

print("\n=== FORCE CHECK ===")
df_final.groupBy("force_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

print("\n=== GRAIN DUPLICATE CHECK ===")
grain_dupes = df_final.groupBy(
    "lsoa_code", "year", "month_num", "force_name"
).count() \
.filter(col("count") > 1)
print(f"Duplicate rows at grain: {grain_dupes.count():,}")

print("\n=== ROW COUNT ===")
print(f"Total rows: {df_final.count():,}")

In [0]:
# ================================================
# CELL 4 - EXPORT AS CSV
# ================================================

df_final.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(OUTPUT_PATH)

print(f"✅ EXPORT COMPLETE")
print(f"✅ Location: {OUTPUT_PATH}")
print(f"✅ Total rows: {df_final.count():,}")

In [0]:
# ================================================
# CELL 5 - VERIFY EXPORT
# ================================================
from pyspark.sql.functions import sum as spark_sum

df_verify = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(OUTPUT_PATH)

print("=== EXPORT VERIFICATION ===")
print(f"✅ Rows in exported file: {df_verify.count():,}")
print(f"✅ Columns: {len(df_verify.columns)}")

print("\n=== YEAR CHECK ===")
df_verify.groupBy("year") \
    .count() \
    .orderBy("year") \
    .show()

print("\n=== FORCE CHECK ===")
df_verify.groupBy("force_name") \
    .count() \
    .orderBy("count", ascending=False) \
    .show(truncate=False)

display(df_verify.limit(10))

In [0]:
# ================================================
# CELL 6 - RENAME FILE
# ================================================

files = dbutils.fs.ls(OUTPUT_PATH)
part_file = [f.path for f in files 
             if f.name.startswith("part-")][0]

print(f"Found file: {part_file}")

dbutils.fs.cp(
    part_file,
    f"/Volumes/{CATALOG}/gold/outputs/crime_data_final_2017_2021.csv"
)

print(f"✅ File saved as: crime_data_final_2017_2021.csv")
print(f"✅ Location: /Volumes/{CATALOG}/gold/outputs/")